# ROGI correlation with regression performance: ADMET landscape roughness and cross-validated model performance.

This notebook computes the ROGI-XD ADMET landscape smoothness across 11 molecular representations and associates
this with cross-validated model performance (5-fold CV). The pipeline is as follows: 

- Load the Biogen data and compute generic Bemis-Murcko scaffold splits
- Compute ROGI-XD for each representation across the entire Biogen dataset
- Perform 5-fold cross-validation and correlate performance across each endpoint and model with ROGI-XD on the full dataset.
- Generate plots

The only section you should need to edit to reproduce the data from the paper is the path definition section below.

## Imports

In [ ]:
import warnings

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.stats import pearsonr, spearmanr

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
)
from sklearn.svm import SVR

from rogi_xd.rogi import rogi
from rogi_xd.utils import Metric, IntegrationDomain

from pathlib import Path

from sklearn.metrics.pairwise import cosine_similarity
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler

from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator


## Path definitions

- Edit the path definitions below

In [ ]:
# Input folder path for molecular embeddings
folder      = Path("path/to/Biogen embeddings")
# Output path for cross-validated model performances
SCAFFOLD_PERFORMANCE_OUTPUT_PATH    = ""
# Output paths for storing ROGI-XD correlation with model performance
ROGIXD_PERFORMANCE_OUTPUT_PATH      = ""
ROGIXD_FULL_PERFORMANCE_OUTPUT_PATH = ""
ROGIXD_CORR_PERFORMANCE_OUTPUT_PATH = ""
# Path to biogen dataset
BIOGEN_PATH = 'path/to/biogen/dataset'
# MolBERT invalid molecules
MOLBERT_INVALID = 'path/to/Biogen embeddings/MolBERT Biogen invalid.npy'
# COATI invalid molecules
COATI_INVALID = 'path/to/Biogen embeddings/COATI Biogen invalid.npy'

## Load Biogen dataset

- Loads the Biogen dataset, filters invalid molecules that fail to be embedded by COATI and MolBERT, and generates molecule objects.

In [ ]:
biogen = pd.read_csv(BIOGEN_PATH)
biogen

,Internal ID,Vendor ID,SMILES,CollectionName,LOG HLM_CLint (mL/min/kg),LOG MDR1-MDCK ER (B-A/A-B),LOG SOLUBILITY PH 6.8 (ug/mL),LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound),LOG PLASMA PROTEIN BINDING (RAT) (% unbound),LOG RLM_CLint (mL/min/kg)
0,Mol1,317714313,CNc1cc(Nc2cccn(-c3ccccn3)c2=O)nn2c(C(=O)N[C@@H...,emolecules,0.675687,1.493167,0.089905,0.991226,0.518514,1.392169
1,Mol2,324056965,CCOc1cc2nn(CCC(C)(C)O)cc2cc1NC(=O)c1cccc(C(F)F)n1,emolecules,0.675687,1.040780,0.550228,0.099681,0.268344,1.027920
2,Mol3,304005766,CN(c1ncc(F)cn1)[C@H]1CCCNC1,emolecules,0.675687,-0.358806,NaN,2.000000,2.000000,1.027920
3,Mol4,194963090,CC(C)(Oc1ccc(-c2cnc(N)c(-c3ccc(Cl)cc3)c2)cc1)C...,emolecules,0.675687,1.026662,1.657056,-1.158015,-1.403403,1.027920
4,Mol5,324059015,CC(C)(O)CCn1cc2cc(NC(=O)c3cccc(C(F)(F)F)n3)c(C...,emolecules,0.996380,1.010597,NaN,1.015611,1.092264,1.629093
...,...,...,...,...,...,...,...,...,...,...
3516,Mol3517,43258693,O=C(c1ccc2c(c1)CCCC2)N1CCOCC1c1ccn[nH]1,emolecules,NaN,0.606813,NaN,NaN,NaN,NaN
3517,Mol3518,27448206,O=C(Nc1nc2ccccc2[nH]1)c1ccc(-n2cccc2)cc1,emolecules,NaN,-0.444495,NaN,NaN,NaN,NaN
3518,Mol3519,207150215,NC(=O)c1noc([C@@H](CCCC2CCCCC2)CC(=O)NO)n1,emolecules,0.863799,NaN,NaN,NaN,NaN,NaN
3519,Mol3520,25037224,CCCCCCCCc1ccc(CC[C@](N)(CO)COP(=O)(O)O)cc1,emolecules,0.881385,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Drop rows containing molecules not parseable by MolBERT or COATI
molbert_invalid = np.load(MOLBERT_INVALID)
coati_invalid   = np.load(COATI_INVALID)
invalid         = (molbert_invalid | coati_invalid).astype(bool)
invalid.sum()

np.int64(6)

In [9]:
biogen = biogen.loc[~invalid]
biogen

,Internal ID,Vendor ID,SMILES,CollectionName,LOG HLM_CLint (mL/min/kg),LOG MDR1-MDCK ER (B-A/A-B),LOG SOLUBILITY PH 6.8 (ug/mL),LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound),LOG PLASMA PROTEIN BINDING (RAT) (% unbound),LOG RLM_CLint (mL/min/kg)
0,Mol1,317714313,CNc1cc(Nc2cccn(-c3ccccn3)c2=O)nn2c(C(=O)N[C@@H...,emolecules,0.675687,1.493167,0.089905,0.991226,0.518514,1.392169
1,Mol2,324056965,CCOc1cc2nn(CCC(C)(C)O)cc2cc1NC(=O)c1cccc(C(F)F)n1,emolecules,0.675687,1.040780,0.550228,0.099681,0.268344,1.027920
2,Mol3,304005766,CN(c1ncc(F)cn1)[C@H]1CCCNC1,emolecules,0.675687,-0.358806,NaN,2.000000,2.000000,1.027920
3,Mol4,194963090,CC(C)(Oc1ccc(-c2cnc(N)c(-c3ccc(Cl)cc3)c2)cc1)C...,emolecules,0.675687,1.026662,1.657056,-1.158015,-1.403403,1.027920
4,Mol5,324059015,CC(C)(O)CCn1cc2cc(NC(=O)c3cccc(C(F)(F)F)n3)c(C...,emolecules,0.996380,1.010597,NaN,1.015611,1.092264,1.629093
...,...,...,...,...,...,...,...,...,...,...
3516,Mol3517,43258693,O=C(c1ccc2c(c1)CCCC2)N1CCOCC1c1ccn[nH]1,emolecules,NaN,0.606813,NaN,NaN,NaN,NaN
3517,Mol3518,27448206,O=C(Nc1nc2ccccc2[nH]1)c1ccc(-n2cccc2)cc1,emolecules,NaN,-0.444495,NaN,NaN,NaN,NaN
3518,Mol3519,207150215,NC(=O)c1noc([C@@H](CCCC2CCCCC2)CC(=O)NO)n1,emolecules,0.863799,NaN,NaN,NaN,NaN,NaN
3519,Mol3520,25037224,CCCCCCCCc1ccc(CC[C@](N)(CO)COP(=O)(O)O)cc1,emolecules,0.881385,NaN,NaN,NaN,NaN,NaN


In [10]:
endpoints = [
    "LOG HLM_CLint (mL/min/kg)",
    "LOG MDR1-MDCK ER (B-A/A-B)",
    "LOG SOLUBILITY PH 6.8 (ug/mL)",
    "LOG RLM_CLint (mL/min/kg)",
]

In [11]:
mols = [Chem.MolFromSmiles(s) for s in biogen['SMILES'].to_numpy()]

## Compute pairwise Tanimoto and cosine matrices for fingerprints, embeddings, and RDKit PDV

- Generates ECFP4, Atom Pair, and Topological Torsion fingerprints for the Biogen dataset. 
- Computes pairwise Tanimoto similarity matrices for these
- Computes a pairwise RDKit cosine similarity matrix after scaling 
- Loads embeddings for the 9 molecular embedding models and computes pairwise cosine similarity matrices.

In [12]:
def tanimoto_matrix(fps):
    """ 
        Computes a pairwise Tanimoto similarity matrix
        for a given set of fingerprints.
    """
    n      = len(fps)
    matrix = np.ones(shape=(n, n))
    for i in range(n - 1):
        ref = fps[i]
        qrs = fps[i + 1:]
        sim = DataStructs.BulkTanimotoSimilarity(ref, qrs)
        matrix[i, i + 1:] = sim
        matrix[i + 1:, i] = sim
    return matrix

In [13]:
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
apgen  = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=1024)
ttgen  = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=1024)

mprints = [mfpgen.GetFingerprint(m) for m in mols]
aprints = [apgen.GetFingerprint(m) for m in mols]
tprints = [ttgen.GetFingerprint(m) for m in mols]

morgan_tanimoto_matrix = tanimoto_matrix(mprints)
atompr_tanimoto_matrix = tanimoto_matrix(aprints)
toptor_tanimoto_matrix = tanimoto_matrix(tprints)

morgan_matrix = np.stack(mprints)
appair_matrix = np.stack(aprints)
ttorsn_matrix = np.stack(tprints)

In [14]:
descriptor_names = [name for name, _ in Descriptors.descList]
calculator = MolecularDescriptorCalculator(descriptor_names)


def build_rdkit_pdv_preprocessor():
    """Create the RDKit descriptor preprocessing pipeline.

    A fresh instance is used for each fitting scope:
    - once on all molecules to define the global PDV geometry used for ROGI-XD
    - separately inside each scaffold-CV training fold for regression/kNN.
    """
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "variance",
            VarianceThreshold(),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ])


# Raw descriptors are retained for cross-validation. The CV code below fits a
# fresh preprocessing pipeline on each training fold and transforms that fold's
# held-out molecules with the fitted training-fold parameters.
pdv_matrix = np.asarray(
    [calculator.CalcDescriptors(mol) for mol in mols],
    dtype=np.float64,
)

# Independently define the GLOBAL RDKit-PDV representation used for ROGI-XD.
# This preprocessing is intentionally fitted to all molecules because ROGI-XD
# is being treated as a descriptive property of the full dataset/representation,
# matching the ROGI-XD authors' comparison setup.
pdv_rogi_preprocessor               = build_rdkit_pdv_preprocessor()
pdv_scaled_global                   = pdv_rogi_preprocessor.fit_transform(pdv_matrix)
rdkit_pdv_cosine_similarity_matrix  = cosine_similarity(pdv_scaled_global)


In [15]:
tan_mats = {
    "ECFP4": morgan_tanimoto_matrix,
    "Atom Pair": atompr_tanimoto_matrix,
    "Torsion": toptor_tanimoto_matrix,
}

In [16]:
# Cosine-similarity matrices that define the global representation geometry used later for the full-dataset ROGI-XD calculation.
cosine_similarity_matrices = {
    "RDKit PDV": rdkit_pdv_cosine_similarity_matrix,
}

for path in sorted(folder.glob("*.npy")):
    molecule_embeddings = np.load(
        path,
        allow_pickle=True,
    )
    # Censor molecules that are not computable for MolBERT or COATI
    molecule_embeddings = molecule_embeddings[~invalid]
    embedding_matrix    = np.stack(molecule_embeddings).astype(float)

    # Compute all molecule-by-molecule cosine similarities.
    similarity_matrix = cosine_similarity(embedding_matrix)

    # Store by name
    cosine_similarity_matrices[path.stem] = similarity_matrix


In [17]:
# Create a dictionary for all similarity matrices
similarity_matrices = tan_mats | cosine_similarity_matrices
similarity_matrices.keys()

# Delete MolE and MiniMol Biogen embeddings
del similarity_matrices['MolE Biogen embeddings']
del similarity_matrices['MiniMol Biogen embeddings']

In [18]:
print(similarity_matrices.keys())

dict_keys(['ECFP4', 'Atom Pair', 'Torsion', 'RDKit PDV', 'COATI Biogen embeddings', 'CheMeleon Biogen embeddings', 'ChemBERTa Biogen embeddings', 'MolBERT Biogen embeddings', 'MolCLR Biogen embeddings', 'MolFormer Biogen embeddings', 'UniMol Biogen embeddings'])


In [19]:
# Dictionary eature matrices used for regression on the Biogen dataset
feature_matrices = {
    "ECFP4": morgan_matrix,
    "Atom Pair": appair_matrix,
    "Torsion": ttorsn_matrix,
    "RDKit PDV": pdv_matrix,
}

# Append learned embeddings to the dictionary
for path in sorted(folder.glob("*.npy")):
    molecule_embeddings = np.load(
        path,
        allow_pickle=True,
    )
    # Censor molecules that are not computable for MolBERT or COATI
    molecule_embeddings = molecule_embeddings[~invalid]
    embedding_matrix    = np.stack(molecule_embeddings).astype(float)

    feature_matrices[path.stem] = embedding_matrix

del feature_matrices['MolE Biogen embeddings']
del feature_matrices['MiniMol Biogen embeddings']

In [20]:
print(feature_matrices.keys())

dict_keys(['ECFP4', 'Atom Pair', 'Torsion', 'RDKit PDV', 'COATI Biogen embeddings', 'CheMeleon Biogen embeddings', 'ChemBERTa Biogen embeddings', 'MolBERT Biogen embeddings', 'MolCLR Biogen embeddings', 'MolFormer Biogen embeddings', 'UniMol Biogen embeddings'])


## Generic Bemis-Murcko scaffold split

- Maps every molecule to its largest fragment by HAC
- Computes a generic Bemis-Murcko scaffold for each SMILES
- Tabulates scaffold statistics

In [21]:
def largest_fragment(molecule):
    """
        Select the fragment with the greatest number of heavy atoms.

        This prevents counterions and small disconnected fragments from
        defining the scaffold.
    """
    fragments = Chem.GetMolFrags(
        molecule,
        asMols=True,
        sanitizeFrags=True,
    )

    if not fragments:
        raise ValueError(
            "The molecule contained no valid fragments."
        )

    return max(
        fragments,
        key=lambda fragment: fragment.GetNumHeavyAtoms(),
    )


def bemis_murcko_group(
    smiles,
    generic=False,
):
    """
    Generate a Bemis-Murcko scaffold-group identifier.

    Parameters
    ----------
    smiles
        Input molecular SMILES.

    generic
        When True, convert scaffold atoms to carbon and scaffold
        bonds to single bonds. This produces larger, more broadly
        defined scaffold groups.

    Notes
    -----
    Molecules with an empty Murcko scaffold, usually acyclic
    molecules, receive a canonical-molecule identifier.

    This prevents structurally distinct acyclic molecules from
    being assigned to the same empty scaffold group.

    Identical acyclic structures retain the same group identifier,
    ensuring they remain together during scaffold cross-validation.
    """

    molecule = Chem.MolFromSmiles(
        str(smiles)
    )

    if molecule is None:
        raise ValueError(
            f"Invalid SMILES: {smiles}"
        )

    molecule = largest_fragment(
        molecule
    )

    scaffold = MurckoScaffold.GetScaffoldForMol(
        molecule
    )

    if generic:
        scaffold = MurckoScaffold.MakeScaffoldGeneric(
            scaffold
        )

    scaffold_smiles = Chem.MolToSmiles(
        scaffold,
        canonical=True,
        isomericSmiles=False,
    )

    # Handle molecules without a Bemis-Murcko scaffold
    if not scaffold_smiles:

        canonical_smiles = Chem.MolToSmiles(
            molecule,
            canonical=True,
            isomericSmiles=False,
        )

        return f"ACYCLIC_{canonical_smiles}"

    return scaffold_smiles



def build_scaffold_groups(df, smiles_column, generic=True,):
    """
        Compute the scaffolds for every molecule.
    """
    groups = []

    for smiles in df[smiles_column]:
     
        group = bemis_murcko_group(
            smiles=smiles,
            generic=generic,
        )
        groups.append(group)

    return np.asarray(groups,dtype=object,)

In [22]:
scaffold_groups = build_scaffold_groups(
    df=biogen,
    smiles_column="SMILES",
    generic=True,
)

In [23]:
scaffold_summary_df = (
    pd.Series(
        scaffold_groups,
        name="scaffold",
    )
    .value_counts()
    .rename_axis("scaffold")
    .reset_index(name="n_molecules")
)

print(
    "Molecules:",
    len(scaffold_groups),
)

print(
    "Unique scaffold groups:",
    scaffold_summary_df.shape[0],
)

print(
    "Largest scaffold group:",
    scaffold_summary_df["n_molecules"].max(),
)

display(
    scaffold_summary_df.head(20)
)

Molecules: 3515
Unique scaffold groups: 1899
Largest scaffold group: 91


,scaffold,n_molecules
0,C1CCC(C2CCCC2)CC1,91
1,C1CCCCC1,76
2,C1CCC(C2CCCCC2)CC1,52
3,C1CCC(CCC2CCCC2)CC1,47
4,C1CCC2CCCC2C1,47
5,C1CCC(CCC2CCCCC2)CC1,43
6,C1CCC(CC2CCCCC2)CC1,39
7,C1CCC(CC2CCCC2)CC1,31
8,CC(CC1CCCCC1)C1CCCCC1,29
9,CC(CCC1CCCCC1)CC1CCCCC1,27


## Set-up and helper functions for ROGI-XD calculations

In [24]:
def convert_similarity_to_distance(
    matrix,
    conversion,
):
    """
    Convert a molecular similarity matrix into a symmetric distance
    matrix bounded between zero and one.
    """
    similarity = np.asarray(
        matrix,
        dtype=float,
    )

    if conversion == "unit_similarity":
        distance = 1.0 - similarity

    elif conversion == "cosine_similarity":
        distance = (
            1.0 - similarity
        ) / 2.0

    else:
        raise ValueError(
            "conversion must be 'unit_similarity' or "
            "'cosine_similarity'."
        )

    distance = np.clip(
        distance,
        0.0,
        1.0,
    )

    distance = (
        distance + distance.T
    ) / 2.0

    np.fill_diagonal(
        distance,
        0.0,
    )

    return distance

In [25]:
fingerprint_representations = {
    "ECFP4",
    "Atom Pair",
    "Torsion",
}

conversion_by_representation = {
    representation: (
        "unit_similarity"
        if representation
        in fingerprint_representations
        else "cosine_similarity"
    )
    for representation
    in similarity_matrices
}

In [26]:
distance_matrices = {
    representation:
        convert_similarity_to_distance(
            matrix=similarity,
            conversion=conversion_by_representation[
                representation
            ],
        )
    for representation, similarity
    in similarity_matrices.items()
}

In [27]:
def calculate_rogi_xd(
    distance_matrix,
    endpoint_values,
    min_dt=1e-4,
):
    """
    Calculate official ROGI-XD from a precomputed distance matrix.
    """
    distance_matrix = np.asarray(
        distance_matrix,
        dtype=float,
    )

    endpoint_values = np.asarray(
        endpoint_values,
        dtype=float,
    )

    result = rogi(
        xs=distance_matrix,
        y=endpoint_values,
        normalize=True,
        metric=Metric.PRECOMPUTED,
        max_dist=1.0,
        min_dt=min_dt,
        domain=IntegrationDomain.LOG_CLUSTER_RATIO,
        nboots=1,
    )

    return float(
        result.rogi
    )

In [ ]:
def to_dense_array(values):
    """ Helper for dense arrays  """
    if sparse.issparse(values):
        return values.toarray()

    return np.asarray(
        values
    )


def prepare_feature_matrix(values):
    """ Helper for feature matrices """
    if isinstance(
        values,
        pd.DataFrame,
    ):
        return values.to_numpy()

    if sparse.issparse(values):
        return values.tocsr()

    return np.asarray(
        values
    )


def cosine_distance_matrix(features):
    """ Return a square cosine-derived distance matrix in [0, 1]."""
    similarity = cosine_similarity(features)

    return convert_similarity_to_distance(
        matrix=similarity,
        conversion="cosine_similarity",
    )


def cosine_cross_distance(query_features, reference_features):
    """ Return query-to-reference cosine-derived distances in [0, 1]."""
    similarity = cosine_similarity(
        query_features,
        reference_features,
    )

    return np.clip(
        (1.0 - similarity) / 2.0,
        0.0,
        1.0,
    )


def build_feature_models(
    input_is_sparse,
    random_state,
):
    """
    Create the four feature-based regression models.

    SVR uses standardized endpoint values internally so their
    fixed hyperparameters are comparable across endpoints.
    """
    if input_is_sparse:
        imputer = SimpleImputer(
            strategy="constant",
            fill_value=0.0,
        )
    else:
        imputer = SimpleImputer(
            strategy="median"
        )

    densifier = FunctionTransformer(
        to_dense_array,
        accept_sparse=True,
        validate=False,
    )

    random_forest = Pipeline([
        (
            "imputer",
            imputer,
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_features="sqrt",
                min_samples_leaf=3,
                random_state=random_state,
                n_jobs=-1,
            ),
        ),
    ])

    gradient_boosting = Pipeline([
        (
            "imputer",
            imputer,
        ),
        (
            "dense",
            densifier,
        ),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.03,
                max_depth=2,
                min_samples_leaf=5,
                subsample=0.8,
                max_features='sqrt',
                random_state=random_state,
            ),
        ),
    ])

    svr_scaler = StandardScaler(
        with_mean=not input_is_sparse
    )

    svr_pipeline = Pipeline([
        (
            "imputer",
            imputer,
        ),
        (
            "scaler",
            svr_scaler,
        ),
        (
            "model",
            SVR(
                kernel="rbf",
                C=10.0,
                epsilon=0.10,
                gamma="scale",
            ),
        ),
    ])

    svr = TransformedTargetRegressor(
        regressor=svr_pipeline,
        transformer=StandardScaler(),
    )

    return {
        "Random forest":
            random_forest,

        "Gradient boosting":
            gradient_boosting,

        "SVR":
            svr,
    }

In [ ]:
def calculate_sem(values):
    """ Calculate standard error of the mean. """
    values = pd.Series(
        values
    ).dropna()

    if len(values) < 2:
        return np.nan

    return float(
        values.std(ddof=1)
        / np.sqrt(len(values))
    )


def calculate_full_dataset_rogi_xd(
    distance_matrices,
    df,
    endpoints,
    representations=None,
    rogi_min_dt=1e-4,
    verbose=True,
):
    """
    Calculate one full-dataset ROGI-XD value per endpoint/representation.

    Each representation first has one global molecule-by-molecule distance
    matrix. For an endpoint with missing target values, the same valid-molecule
    subset is taken from every representation's global distance matrix before
    calculating ROGI-XD. 
    """

    if representations is None:
        representations = list(distance_matrices.keys())

    representations = list(representations)
    rows = []

    for endpoint in endpoints:
        endpoint_all = pd.to_numeric(
            df[endpoint],
            errors="coerce",
        ).to_numpy(dtype=float)

        valid_mask    = np.isfinite(endpoint_all)
        valid_indices = np.flatnonzero(valid_mask)
        y = endpoint_all[valid_indices]

        if verbose:
            print(f"\nROGI-XD: {endpoint} ({len(y)} valid molecules)")

        for representation in representations:
            if representation not in distance_matrices:
                raise KeyError(
                    f"No global distance matrix for '{representation}'."
                )

            full_distance = np.asarray(
                distance_matrices[representation],
                dtype=float,
            )

            # Apply the same endpoint-valid subset to every representation.
            endpoint_distance = full_distance[
                np.ix_(valid_indices, valid_indices)
            ]

            rogi_value = calculate_rogi_xd(
                distance_matrix=endpoint_distance,
                endpoint_values=y,
                min_dt=rogi_min_dt,
            )

            rows.append({
                "endpoint": endpoint,
                "representation": representation,
                "n_valid": len(y),
                "rogi_xd": rogi_value,
            })

            if verbose:
                print(
                    f"  {representation}: ROGI-XD = {rogi_value:.4f}"
                )

    return pd.DataFrame(rows)


def evaluate_scaffold_cv(
    feature_matrices,
    distance_matrices,
    scaffold_groups,
    df,
    endpoints,
    representations=None,
    models_to_run=(
        "Random forest",
        "Gradient boosting",
        "SVR",
        "kNN",
    ),
    n_outer_splits=5,
    knn_k=5,
    knn_weights="distance",
    random_state=42,
    verbose=True,
):
    """
    Evaluate molecular representations using scaffold-group CV.

    Predictive performance is measured on held-out scaffold groups.
    
    RDKit PDV preprocessing is fitted independently inside each training fold,
    preventing test-fold information from entering variance filtering or
    standardization. The same fold-fitted transform is used for RDKit-PDV
    feature models and its kNN distance matrices.

    Returns
    -------
    performance_summary_df
        Mean performance across scaffold folds.

    performance_fold_df
        One row per endpoint, representation, model and fold.
    """
    expected_n = len(df)

    scaffold_groups = np.asarray(
        scaffold_groups,
        dtype=object,
    )

    if representations is None:
        representations = list(feature_matrices.keys())

    representations = list(representations)

    prepared_features = {
        representation: prepare_feature_matrix(
            feature_matrices[representation]
        )
        for representation in representations
        if representation in feature_matrices
    }

    fold_rows = []

    for endpoint_index, endpoint in enumerate(endpoints):
        endpoint_all = pd.to_numeric(
            df[endpoint],
            errors="coerce",
        ).to_numpy(dtype=float)

        valid_mask = np.isfinite(endpoint_all)
        valid_indices = np.flatnonzero(valid_mask)
        y = endpoint_all[valid_indices]
        groups = scaffold_groups[valid_indices]

        number_of_groups = len(np.unique(groups))
        endpoint_n_splits = min(
            int(n_outer_splits),
            number_of_groups,
        )

        outer_cv = GroupKFold(
            n_splits=endpoint_n_splits
        )

        outer_splits = list(
            outer_cv.split(
                X=np.zeros(len(y)),
                y=y,
                groups=groups,
            )
        )

        if verbose:
            print(f"\n{endpoint}")
            print(f"  valid molecules: {len(y)}")
            print(f"  scaffold groups: {number_of_groups}")
            print(f"  folds: {endpoint_n_splits}")

        for fold_index, (train_indices, test_indices) in enumerate(
            outer_splits
        ):
            train_groups = set(groups[train_indices])
            test_groups  = set(groups[test_indices])

            y_train = y[train_indices]
            y_test = y[test_indices]

            training_sd = float(
                np.std(y_train, ddof=1)
            )

            baseline_prediction = np.full(
                len(y_test),
                np.mean(y_train),
                dtype=float,
            )

            baseline_rmse = float(
                np.sqrt(
                    mean_squared_error(
                        y_test,
                        baseline_prediction,
                    )
                )
            )

            if verbose:
                print(
                    f"  fold {fold_index + 1}: "
                    f"{len(train_indices)} train / "
                    f"{len(test_indices)} test; "
                    f"{len(train_groups)} train scaffolds / "
                    f"{len(test_groups)} test scaffolds"
                )

            for representation_index, representation in enumerate(
                representations
            ):
                endpoint_features = None
                x_train = None
                x_test = None

                if representation in prepared_features:
                    endpoint_features = prepared_features[
                        representation
                    ][valid_indices]

                if representation == "RDKit PDV":
                    raw_x_train = endpoint_features[train_indices]
                    raw_x_test = endpoint_features[test_indices]

                    pdv_preprocessor = build_rdkit_pdv_preprocessor()
                    x_train = pdv_preprocessor.fit_transform(raw_x_train)
                    x_test = pdv_preprocessor.transform(raw_x_test)

                    # kNN uses distances constructed from the same
                    # training-fold-fitted descriptor transform.
                    train_distance = cosine_distance_matrix(x_train)
                    test_to_train_distance = cosine_cross_distance(
                        x_test,
                        x_train,
                    )
                    input_is_sparse = False

                else:
                    full_distance = np.asarray(
                        distance_matrices[representation],
                        dtype=float,
                    )

                    endpoint_distance = full_distance[
                        np.ix_(valid_indices, valid_indices)
                    ]

                    train_distance = endpoint_distance[
                        np.ix_(train_indices, train_indices)
                    ]

                    test_to_train_distance = endpoint_distance[
                        np.ix_(test_indices, train_indices)
                    ]

                    if endpoint_features is not None:
                        x_train = endpoint_features[train_indices]
                        x_test = endpoint_features[test_indices]
                        input_is_sparse = sparse.issparse(
                            endpoint_features
                        )
                    else:
                        input_is_sparse = False

                model_seed = (
                    random_state
                    + endpoint_index * 100_000
                    + fold_index * 1_000
                    + representation_index
                )

                feature_models = build_feature_models(
                    input_is_sparse=input_is_sparse,
                    random_state=model_seed,
                )

                for model_name in models_to_run:
                    if verbose:
                        print(
                            f"    {representation} — {model_name}"
                        )

                    if model_name == "kNN":
                        effective_k = min(
                            int(knn_k),
                            len(train_indices),
                        )

                        estimator = KNeighborsRegressor(
                            n_neighbors=effective_k,
                            weights=knn_weights,
                            metric="precomputed",
                            algorithm="brute",
                            n_jobs=-1,
                        )

                        estimator.fit(
                            train_distance,
                            y_train,
                        )

                        prediction = estimator.predict(
                            test_to_train_distance
                        )

                    else:
                        if x_train is None or x_test is None:
                            raise ValueError(
                                f"Feature matrix required for "
                                f"'{representation}' and "
                                f"'{model_name}'."
                            )

                        estimator = feature_models[model_name]

                        with warnings.catch_warnings():
                            warnings.filterwarnings(
                                "ignore",
                                category=ConvergenceWarning,
                            )

                            estimator.fit(
                                x_train,
                                y_train,
                            )

                        prediction = estimator.predict(x_test)

                    rmse = float(
                        np.sqrt(
                            mean_squared_error(
                                y_test,
                                prediction,
                            )
                        )
                    )

                    mae = float(
                        mean_absolute_error(
                            y_test,
                            prediction,
                        )
                    )

                    r2 = float(
                        r2_score(
                            y_test,
                            prediction,
                        )
                    )

                    fold_rows.append({
                        "endpoint": endpoint,
                        "representation": representation,
                        "model": model_name,
                        "fold": fold_index,
                        "n_valid": len(y),
                        "n_train": len(train_indices),
                        "n_test": len(test_indices),
                        "n_train_scaffolds": len(train_groups),
                        "n_test_scaffolds": len(test_groups),
                        "rmse": rmse,
                        "rmse_over_train_sd": rmse / training_sd,
                        "mae": mae,
                        "mae_over_train_sd": mae / training_sd,
                        "r2": r2,
                        "baseline_rmse": baseline_rmse,
                        "baseline_rmse_over_train_sd": (
                            baseline_rmse / training_sd
                        ),
                    })

    performance_fold_df = pd.DataFrame(fold_rows)

    performance_summary_df = (
        performance_fold_df
        .groupby(
            [
                "endpoint",
                "representation",
                "model",
            ],
            as_index=False,
        )
        .agg(
            n_valid=("n_valid", "first"),
            n_folds=("fold", "nunique"),
            rmse=("rmse", "mean"),
            rmse_sem=("rmse", calculate_sem),
            rmse_over_train_sd=("rmse_over_train_sd", "mean"),
            rmse_over_train_sd_sem=(
                "rmse_over_train_sd",
                calculate_sem,
            ),
            mae=("mae", "mean"),
            r2=("r2", "mean"),
            baseline_rmse_over_train_sd=(
                "baseline_rmse_over_train_sd",
                "mean",
            ),
        )
    )

    return (
        performance_summary_df,
        performance_fold_df,
    )

In [31]:
scaffold_performance_df, scaffold_fold_df = (
    evaluate_scaffold_cv(
        feature_matrices=feature_matrices,
        distance_matrices=distance_matrices,
        scaffold_groups=scaffold_groups,
        df=biogen,
        endpoints=endpoints,
        models_to_run=(
            "kNN",
            "Random forest",
            "SVR",
            "Gradient boosting",
        ),
        n_outer_splits=5,
        knn_k=5,
        knn_weights="distance",
        random_state=42,
        verbose=True,
    )
)

# One full-dataset ROGI-XD value per endpoint/representation, calculated
# independently of the scaffold-CV folds. Every representation, including
# RDKit PDV, is defined globally first and then subset to molecules with a
# valid target for the endpoint (matching the ROGI-XD comparison setup).
full_rogi_xd_df = calculate_full_dataset_rogi_xd(
    distance_matrices=distance_matrices,
    df=biogen,
    endpoints=endpoints,
    representations=scaffold_performance_df["representation"].unique(),
    rogi_min_dt=1e-4,
    verbose=True,
)

rogi_performance_df = scaffold_performance_df.merge(
    full_rogi_xd_df[["endpoint", "representation", "rogi_xd"]],
    on=["endpoint", "representation"],
    how="left",
    validate="many_to_one",
)

rogi_performance_df


LOG HLM_CLint (mL/min/kg)
  valid molecules: 3082
  scaffold groups: 1681
  folds: 5
  fold 1: 2465 train / 617 test; 1345 train scaffolds / 336 test scaffolds
    ECFP4 — kNN
    ECFP4 — Random forest
    ECFP4 — SVR
    ECFP4 — Gradient boosting
    Atom Pair — kNN
    Atom Pair — Random forest
    Atom Pair — SVR
    Atom Pair — Gradient boosting
    Torsion — kNN
    Torsion — Random forest
    Torsion — SVR
    Torsion — Gradient boosting
    RDKit PDV — kNN
    RDKit PDV — Random forest
    RDKit PDV — SVR
    RDKit PDV — Gradient boosting
    COATI Biogen embeddings — kNN
    COATI Biogen embeddings — Random forest
    COATI Biogen embeddings — SVR
    COATI Biogen embeddings — Gradient boosting
    CheMeleon Biogen embeddings — kNN
    CheMeleon Biogen embeddings — Random forest
    CheMeleon Biogen embeddings — SVR
    CheMeleon Biogen embeddings — Gradient boosting
    ChemBERTa Biogen embeddings — kNN
    ChemBERTa Biogen embeddings — Random forest
    ChemBERTa Biogen embe

,endpoint,representation,model,n_valid,n_folds,rmse,rmse_sem,rmse_over_train_sd,rmse_over_train_sd_sem,mae,r2,baseline_rmse_over_train_sd,rogi_xd
0,LOG HLM_CLint (mL/min/kg),Atom Pair,Gradient boosting,3082,5,0.560272,0.006218,0.898562,0.010764,0.473211,0.191068,1.000049,0.279577
1,LOG HLM_CLint (mL/min/kg),Atom Pair,Random forest,3082,5,0.546966,0.006594,0.877223,0.011327,0.462686,0.228995,1.000049,0.279577
2,LOG HLM_CLint (mL/min/kg),Atom Pair,SVR,3082,5,0.518391,0.006052,0.831393,0.010437,0.419555,0.307430,1.000049,0.279577
3,LOG HLM_CLint (mL/min/kg),Atom Pair,kNN,3082,5,0.570957,0.008390,0.915674,0.013655,0.471193,0.159249,1.000049,0.279577
4,LOG HLM_CLint (mL/min/kg),COATI Biogen embeddings,Gradient boosting,3082,5,0.571924,0.003932,0.917242,0.007328,0.489097,0.157195,1.000049,0.315197
...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,LOG SOLUBILITY PH 6.8 (ug/mL),Torsion,kNN,2170,5,0.651420,0.006884,0.952915,0.013107,0.474272,0.085174,1.000353,0.275088
172,LOG SOLUBILITY PH 6.8 (ug/mL),UniMol Biogen embeddings,Gradient boosting,2170,5,0.621478,0.007737,0.909237,0.015792,0.451764,0.168855,1.000353,0.277899
173,LOG SOLUBILITY PH 6.8 (ug/mL),UniMol Biogen embeddings,Random forest,2170,5,0.620630,0.008007,0.908001,0.016147,0.458423,0.171142,1.000353,0.277899
174,LOG SOLUBILITY PH 6.8 (ug/mL),UniMol Biogen embeddings,SVR,2170,5,0.605480,0.004489,0.885753,0.011004,0.448987,0.210513,1.000353,0.277899


In [ ]:
scaffold_performance_df.to_csv(SCAFFOLD_PERFORMANCE_OUTPUT_PATH)

### ROGI-XD versus cross-validated performance

- For each endpoint and molecular representation, `full_rogi_xd_df` contains **one ROGI-XD value calculated on the full endpoint dataset**. 
- Cross-validated predictive performance is the **mean RMSE across the five scaffold folds**. 
- The final Pearson correlation is then calculated **across representations for each endpoint × model combination**. Individual folds are not treated as independent correlation observations.

In [33]:
correlation_rows = []

for (endpoint, model_name), group in rogi_performance_df.groupby(
    ["endpoint", "model"],
    sort=False,
):
    group = group.dropna(
        subset=["rogi_xd", "rmse"]
    )

    if group["representation"].duplicated().any():
        raise ValueError(
            f"Duplicate representation rows found for {endpoint} / {model_name}."
        )

    if len(group) < 3:
        pearson_r = np.nan
        pearson_p = np.nan
    else:
        pearson_r, pearson_p = pearsonr(
            group["rogi_xd"],
            group["rmse"],
        )

    correlation_rows.append({
        "endpoint": endpoint,
        "model": model_name,
        "n_representations": len(group),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
    })

rogi_correlation_df = pd.DataFrame(correlation_rows)
rogi_correlation_df

,endpoint,model,n_representations,pearson_r,pearson_p
0,LOG HLM_CLint (mL/min/kg),Gradient boosting,11,0.854258,0.000814
1,LOG HLM_CLint (mL/min/kg),Random forest,11,0.918110,0.000067
2,LOG HLM_CLint (mL/min/kg),SVR,11,0.637992,0.034674
3,LOG HLM_CLint (mL/min/kg),kNN,11,0.774872,0.005095
4,LOG MDR1-MDCK ER (B-A/A-B),Gradient boosting,11,0.880692,0.000344
5,LOG MDR1-MDCK ER (B-A/A-B),Random forest,11,0.919974,0.000060
6,LOG MDR1-MDCK ER (B-A/A-B),SVR,11,0.825616,0.001746
7,LOG MDR1-MDCK ER (B-A/A-B),kNN,11,0.876270,0.000403
8,LOG RLM_CLint (mL/min/kg),Gradient boosting,11,0.738771,0.009401
9,LOG RLM_CLint (mL/min/kg),Random forest,11,0.823237,0.001849


In [34]:
full_rogi_xd_df

,endpoint,representation,n_valid,rogi_xd
0,LOG HLM_CLint (mL/min/kg),Atom Pair,3082,0.279577
1,LOG HLM_CLint (mL/min/kg),COATI Biogen embeddings,3082,0.315197
2,LOG HLM_CLint (mL/min/kg),CheMeleon Biogen embeddings,3082,0.289810
3,LOG HLM_CLint (mL/min/kg),ChemBERTa Biogen embeddings,3082,0.326481
4,LOG HLM_CLint (mL/min/kg),ECFP4,3082,0.316375
5,LOG HLM_CLint (mL/min/kg),MolBERT Biogen embeddings,3082,0.287703
6,LOG HLM_CLint (mL/min/kg),MolCLR Biogen embeddings,3082,0.311802
7,LOG HLM_CLint (mL/min/kg),MolFormer Biogen embeddings,3082,0.299971
8,LOG HLM_CLint (mL/min/kg),RDKit PDV,3082,0.251838
9,LOG HLM_CLint (mL/min/kg),Torsion,3082,0.302157


In [ ]:
rogi_performance_df.to_csv(ROGIXD_PERFORMANCE_OUTPUT_PATH)
full_rogi_xd_df.to_csv(ROGIXD_FULL_PERFORMANCE_OUTPUT_PATH)
rogi_correlation_df.to_csv(ROGIXD_CORR_PERFORMANCE_OUTPUT_PATH)

## Plotting code

In [ ]:
"""
Combined ROGI-XD, RMSE and Pearson correlation figure.

Expected DataFrames to be defined before running:
    full_rogi_xd_df
    scaffold_performance_df
    rogi_correlation_df (including a `pearson_r` column)
"""

# ============================================================
# GLOBAL STYLING
# ============================================================
TOP_N      = 5
PLOT_ALPHA = 0.5  

GLOBAL_FONT        = 20
SUBPLOT_TITLE_FONT = 24
AXIS_TITLE_FONT    = 21
AXIS_TICK_FONT     = 20
BAR_TEXT_FONT      = 21
LEGEND_FONT        = 22
LEGEND_TITLE_FONT  = 22
HEATMAP_VALUE_FONT = 28
HEATMAP_X_FONT     = 27
HEATMAP_Y_FONT     = 23
PANEL_LETTER_FONT  = 34
FIG_WIDTH          = 2250
FIG_HEIGHT         = 1500

endpoint_labels = {
    "LOG HLM_CLint (mL/min/kg)":     "HLM CLint",
    "LOG MDR1-MDCK ER (B-A/A-B)":    "MDR1-MDCK ER",
    "LOG SOLUBILITY PH 6.8 (ug/mL)": "Solubility",
    "LOG RLM_CLint (mL/min/kg)":     "RLM CLint",
}

endpoint_order = ["HLM CLint", "RLM CLint", "MDR1-MDCK ER", "Solubility"]
model_order    = ["kNN", "Random forest", "Gradient boosting", "SVR"]

corr_endpoint_labels = {
    "HLM CLint":    "HLM CLint",
    "RLM CLint":    "RLM CLint",
    "MDR1-MDCK ER": "MDR1<br>MDCK ER",
    "Solubility":   "Solubility",
}


# ============================================================
# COLOUR UTILITIES
# ============================================================
def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4))


def rgb_to_hex(rgb):
    return "#{:02X}{:02X}{:02X}".format(*rgb)


def hex_to_rgba(hex_color, alpha=1.0):
    r, g, b = hex_to_rgb(hex_color)
    return f"rgba({r},{g},{b},{alpha})"


def interpolate_from_stops(value, vmin, vmax, color_stops):
    if vmax == vmin:
        return color_stops[-1][1]
    t = np.clip((value - vmin) / (vmax - vmin), 0, 1)
    for (p0, c0), (p1, c1) in zip(color_stops[:-1], color_stops[1:]):
        if p0 <= t <= p1:
            local_t = (t - p0) / (p1 - p0)
            rgb0, rgb1 = hex_to_rgb(c0), hex_to_rgb(c1)
            rgb = tuple(round(a + local_t * (b - a)) for a, b in zip(rgb0, rgb1))
            return rgb_to_hex(rgb)
    return color_stops[-1][1]


# ============================================================
# PANEL A DATA: ROGI-XD SMOOTHNESS
# ============================================================
smooth_df = full_rogi_xd_df.copy()
smooth_df["endpoint_short"] = (
    smooth_df["endpoint"].map(endpoint_labels).fillna(smooth_df["endpoint"])
)
smooth_df["representation_label"] = (
    smooth_df["representation"].str.replace(" Biogen embeddings", "", regex=False)
    + "\u00A0\u00A0"
)
smooth_df = (
    smooth_df.sort_values(["endpoint_short", "rogi_xd"])
    .groupby("endpoint_short", group_keys=False)
    .head(TOP_N)
    .copy()
)
smooth_min = smooth_df["rogi_xd"].min()
smooth_max = smooth_df["rogi_xd"].max()
smooth_color_stops = [
    (0.00, "#B7D3EA"),
    (0.45, "#6FA8B6"),
    (0.75, "#3B6EA8"),
    (1.00, "#1F3A5F"),
]


# ============================================================
# PANEL C DATA: PEARSON CORRELATION
# ============================================================
corr_df = rogi_correlation_df.copy()

corr_df["endpoint_short"] = (
    corr_df["endpoint"].map(endpoint_labels).fillna(corr_df["endpoint"])
)
corr_matrix = (
    corr_df[corr_df["model"].isin(model_order)]
    .pivot(index="endpoint_short", columns="model", values="pearson_r")
    .reindex(index=endpoint_order, columns=model_order)
)
corr_min = np.nanmin(corr_matrix.values)
corr_max = np.nanmax(corr_matrix.values)
corr_color_stops = [
    (0.00, "#1F3A5F"),
    (0.45, "#3B6EA8"),
    (0.75, "#6FA8B6"),
    (1.00, "#B7D3EA"),
]


# ============================================================
# PANEL B DATA: RMSE BY REPRESENTATION
# ============================================================
rmse_df = scaffold_performance_df.copy()
rmse_df["endpoint_short"] = (
    rmse_df["endpoint"].map(endpoint_labels).fillna(rmse_df["endpoint"])
)
rmse_df = rmse_df[rmse_df["model"].isin(model_order)].copy()
rmse_df["representation_label"] = rmse_df["representation"].str.replace(
    " Biogen embeddings", "", regex=False
)
# For each endpoint separately, rank representations by the median RMSE
# across its four machine-learning models. Lower RMSE is better.
# First summarise each model equally (also handles repeated RMSE rows per model).
model_rmse = (
    rmse_df.groupby(["endpoint_short", "representation", "model"], as_index=False)["rmse"]
    .median()
)
endpoint_rep_medians = (
    model_rmse.groupby(["endpoint_short", "representation"], as_index=False)["rmse"]
    .median()
    .rename(columns={"rmse": "median_rmse"})
)

# Select the top five representations within each endpoint.
endpoint_top_reps = (
    endpoint_rep_medians.sort_values(
        ["endpoint_short", "median_rmse", "representation"],
        ascending=[True, True, True],
        kind="stable",
    )
    .groupby("endpoint_short", group_keys=False)
    .head(TOP_N)
)
rmse_df = rmse_df.merge(
    endpoint_top_reps[["endpoint_short", "representation"]],
    on=["endpoint_short", "representation"],
    how="inner",
)

# Keep colours consistent for a representation that appears in multiple
# endpoint plots, even when each endpoint has a different top five.
colour_palette = [
    "#B7D3EA", "#6FA8B6", "#7895A8", "#3B6EA8", "#1F3A5F",
    "#8BB7C6", "#9DB3C8", "#B2CDD8", "#6C95A8", "#C5D8DF",
    "#84AEB8", "#7E9BB2", "#A8C0CF", "#5E8B9F", "#ABCACD",
]
selected_rep_labels = sorted(rmse_df["representation_label"].unique())
rmse_color_map = {
    rep: colour_palette[i % len(colour_palette)]
    for i, rep in enumerate(selected_rep_labels)
}

# Use a pale blue for UniMol in every endpoint panel.
for rep in selected_rep_labels:
    if "unimol" in rep.lower().replace("-", "").replace(" ", ""):
        rmse_color_map[rep] = "#A5C9D4"
model_symbols = {
    "kNN": "circle",
    "Random forest": "square",
    "Gradient boosting": "diamond",
    "SVR": "triangle-up",
}
model_offsets = {
    "kNN": -0.18,
    "Random forest": -0.06,
    "Gradient boosting": 0.06,
    "SVR": 0.18,
}
# ============================================================
# COMBINED FIGURE
# a = ROGI-XD; b = RMSE; c = Pearson correlation
# ============================================================
fig = make_subplots(
    rows=3,
    cols=4,
    specs=[
        [{}, {}, {}, {}],
        [{}, {}, {}, {}],
        [{"colspan": 4}, None, None, None],
    ],
    subplot_titles=[
        "<b>HLM CLint</b>",
        "<b>RLM CLint</b>",
        "<b>HLM CLint</b>",
        "<b>RLM CLint</b>",
        "<b>MDR1-MDCK ER</b>",
        "<b>Solubility</b>",
        "<b>MDR1-MDCK ER</b>",
        "<b>Solubility</b>",
        "",
    ],
    horizontal_spacing=0.020,
    vertical_spacing=0.14,
    row_heights=[0.30, 0.30, 0.40],
)


# ============================================================
# PANEL A: SQUARE-CORNERED BAR CHARTS, BLACK VALUES
# ============================================================
panel_a_positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
for endpoint, (row, col) in zip(endpoint_order, panel_a_positions):
    d = (
        smooth_df[smooth_df["endpoint_short"] == endpoint]
        .sort_values("rogi_xd", ascending=True)
        .copy()
    )
    bar_colors_hex = [
        interpolate_from_stops(value, smooth_min, smooth_max, smooth_color_stops)
        for value in d["rogi_xd"]
    ]
    bar_colors = [hex_to_rgba(color, PLOT_ALPHA) for color in bar_colors_hex]
    rep_labels = d["representation_label"].str.strip()

    fig.add_trace(
        go.Bar(
            x=rep_labels,
            y=d["rogi_xd"],
            orientation="v",
            marker=dict(
                color=bar_colors,
                line=dict(color="#111111", width=1.2),
            ),
            text=d["rogi_xd"].map(lambda x: f"<b>{x:.3f}</b>"),
            textposition="inside",
            insidetextanchor="end",
            textfont=dict(size=BAR_TEXT_FONT, color="#000000"),
            cliponaxis=False,
            hovertemplate="<b>%{x}</b><br>ROGI-XD = %{y:.4f}<extra></extra>",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    fig.update_xaxes(
        tickfont=dict(size=AXIS_TICK_FONT),
        tickangle=-28,
        categoryorder="array",
        categoryarray=d["representation_label"].tolist(),
        showgrid=False,
        zeroline=False,
        showline=False,
        ticks="",
        row=row,
        col=col,
    )
    fig.update_yaxes(
        title_text="ROGI-XD" if col == 1 else "",
        title_font=dict(size=AXIS_TITLE_FONT),
        tickfont=dict(size=AXIS_TICK_FONT),
        showgrid=False,
        zeroline=False,
        showline=False,
        ticks="",
        row=row,
        col=col,
    )


# ============================================================
# PANEL B: RMSE BOXPLOTS AND MODEL MARKERS
# ============================================================
panel_b_positions = [(1, 3), (1, 4), (2, 3), (2, 4)]
for i, (endpoint, (row, col)) in enumerate(zip(endpoint_order, panel_b_positions)):
    endpoint_df = rmse_df[rmse_df["endpoint_short"] == endpoint].copy()

    # The five selected representations are ordered left to right by their
    # endpoint-specific median RMSE (lowest = best).
    endpoint_ranked_reps = endpoint_top_reps[
        endpoint_top_reps["endpoint_short"] == endpoint
    ].sort_values(["median_rmse", "representation"], kind="stable")
    endpoint_rep_order = [
        endpoint_df.loc[
            endpoint_df["representation"] == rep, "representation_label"
        ].iloc[0]
        for rep in endpoint_ranked_reps["representation"]
    ]
    rep_positions = {rep: pos for pos, rep in enumerate(endpoint_rep_order)}

    for rep in endpoint_rep_order:
        d = endpoint_df[endpoint_df["representation_label"] == rep]
        fig.add_trace(
            go.Box(
                x=[rep_positions[rep]] * len(d),
                y=d["rmse"],
                boxpoints=False,
                width=0.58,
                line=dict(color="#000000", width=1.3),
                fillcolor=hex_to_rgba(rmse_color_map[rep], PLOT_ALPHA),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )

    for model in model_order:
        d = endpoint_df[endpoint_df["model"] == model].copy()
        x_values = [
            rep_positions[rep] + model_offsets[model]
            for rep in d["representation_label"]
        ]
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=d["rmse"],
                mode="markers",
                marker=dict(
                    symbol=model_symbols[model],
                    size=10,
                    color=f"rgba(255,255,255,{PLOT_ALPHA})",
                    line=dict(color="#3B4248", width=1.35),
                ),
                customdata=d[["representation_label"]],
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    + model
                    + "<br>RMSE = %{y:.3f}<extra></extra>"
                ),
                name=model,
                legendgroup=model,
                showlegend=(i == 0),
            ),
            row=row,
            col=col,
        )

    fig.update_xaxes(
        range=[-0.6, len(endpoint_rep_order) - 0.4],
        tickmode="array",
        tickvals=list(range(len(endpoint_rep_order))),
        ticktext=endpoint_rep_order,
        tickfont=dict(size=AXIS_TICK_FONT),
        tickangle=-24,
        showgrid=False,
        zeroline=False,
        showline=False,
        ticks="",
        row=row,
        col=col,
    )
    fig.update_yaxes(
        title_text="RMSE" if col == 3 else "",
        title_font=dict(size=AXIS_TITLE_FONT),
        tickfont=dict(size=AXIS_TICK_FONT),
        showgrid=False,
        zeroline=False,
        showline=False,
        row=row,
        col=col,
    )


# ============================================================
# PANEL C: RECTANGULAR PEARSON CORRELATION CELLS
# ============================================================
heatmap_gap = 0.055
n_corr_rows, n_corr_cols = corr_matrix.shape

for row_idx in range(n_corr_rows):
    for col_idx in range(n_corr_cols):
        r_value = corr_matrix.iloc[row_idx, col_idx]
        if pd.isna(r_value):
            continue

        x0, x1 = col_idx + heatmap_gap, col_idx + 1 - heatmap_gap
        y0, y1 = row_idx + heatmap_gap, row_idx + 1 - heatmap_gap
        fill_hex = interpolate_from_stops(
            r_value, corr_min, corr_max, corr_color_stops
        )

        fig.add_shape(
            type="rect",
            x0=x0,
            y0=y0,
            x1=x1,
            y1=y1,
            fillcolor=hex_to_rgba(fill_hex, PLOT_ALPHA),
            line=dict(color="#111111", width=1.2),
            layer="below",
            row=3,
            col=1,
        )
        fig.add_annotation(
            x=col_idx + 0.5,
            y=row_idx + 0.5,
            text=f"<b>{r_value:.2f}</b>",
            showarrow=False,
            font=dict(size=HEATMAP_VALUE_FONT, color="#000000"),
            row=3,
            col=1,
        )

fig.update_xaxes(
    range=[0, n_corr_cols],
    tickmode="array",
    tickvals=np.arange(n_corr_cols) + 0.5,
    ticktext=[f"<b>{x}</b>" for x in corr_matrix.columns],
    side="top",
    showgrid=False,
    zeroline=False,
    showline=False,
    ticks="",
    tickfont=dict(size=HEATMAP_X_FONT),
    ticklabelstandoff=12,
    row=3,
    col=1,
)
fig.update_yaxes(
    range=[n_corr_rows, 0],
    tickmode="array",
    tickvals=np.arange(n_corr_rows) + 0.5,
    ticktext=[f"<b>{corr_endpoint_labels[x]}</b>" for x in corr_matrix.index],
    showgrid=False,
    zeroline=False,
    showline=False,
    ticks="",
    tickfont=dict(size=HEATMAP_Y_FONT),
    row=3,
    col=1,
)


# ============================================================
# PEARSON COLOURBAR
# ============================================================
c_subplot = fig.get_subplot(3, 1)
c_y0, c_y1 = c_subplot.yaxis.domain
c_colorbar_y = (c_y0 + c_y1) / 2
c_colorbar_len = (c_y1 - c_y0) * 0.92
corr_colorscale = [
    [position, hex_to_rgba(color, PLOT_ALPHA)]
    for position, color in corr_color_stops
]
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            size=0.1,
            color=[corr_min],
            cmin=corr_min,
            cmax=corr_max,
            colorscale=corr_colorscale,
            showscale=True,
            colorbar=dict(
                orientation="v",
                title=dict(
                    text="<b>Pearson r</b>",
                    side="top",
                    font=dict(size=22),
                ),
                thickness=18,
                len=c_colorbar_len,
                x=1.005,
                xanchor="left",
                y=c_colorbar_y,
                yanchor="middle",
                outlinewidth=0,
                tickfont=dict(size=20),
            ),
        ),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=3,
    col=1,
)

b_axis_names = ["xaxis3", "xaxis4", "xaxis7", "xaxis8"]
b_old_left = min(fig.layout[name].domain[0] for name in b_axis_names)
b_old_right = max(fig.layout[name].domain[1] for name in b_axis_names)
B_SHIFT = 0.025
b_new_left = b_old_left + B_SHIFT
b_new_right = 1.0


def remap_b_domain(domain):
    old_start, old_end = domain
    old_width = b_old_right - b_old_left
    new_width = b_new_right - b_new_left
    new_start = b_new_left + ((old_start - b_old_left) / old_width) * new_width
    new_end = b_new_left + ((old_end - b_old_left) / old_width) * new_width
    return [new_start, new_end]


for axis_name in b_axis_names:
    fig.layout[axis_name].domain = remap_b_domain(list(fig.layout[axis_name].domain))


# ============================================================
# SUBPLOT TITLES AND PANEL LABELS
# ============================================================

endpoint_title_set = {f"<b>{x}</b>" for x in endpoint_order}
b_title_annotation_indices = [2, 3, 6, 7]
for annotation_index, axis_name in zip(b_title_annotation_indices, b_axis_names):
    domain = fig.layout[axis_name].domain
    fig.layout.annotations[annotation_index].x = (domain[0] + domain[1]) / 2

for annotation in fig.layout.annotations:
    if annotation.text in endpoint_title_set:
        annotation.update(font=dict(size=SUBPLOT_TITLE_FONT), yshift=10)

b_legend_center = (b_new_left + b_new_right) / 2
c_subplot = fig.get_subplot(3, 1)
c_y0, c_y1 = c_subplot.yaxis.domain
TOP_PANEL_LETTER_Y = 1.045

fig.add_annotation(
    x=-0.030,
    y=TOP_PANEL_LETTER_Y,
    xref="paper",
    yref="paper",
    text="<b>a</b>",
    showarrow=False,
    font=dict(size=PANEL_LETTER_FONT),
)
fig.add_annotation(
    x=b_new_left - 0.026,
    y=TOP_PANEL_LETTER_Y,
    xref="paper",
    yref="paper",
    text="<b>b</b>",
    showarrow=False,
    font=dict(size=PANEL_LETTER_FONT),
)
fig.add_annotation(
    x=-0.030,
    y=c_y1 + 0.035,
    xref="paper",
    yref="paper",
    text="<b>c</b>",
    showarrow=False,
    font=dict(size=PANEL_LETTER_FONT),
)


# ============================================================
# OVERALL LAYOUT
# ============================================================
fig.update_layout(
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    plot_bgcolor="white",
    paper_bgcolor="white",
    barcornerradius=0, 
    bargap=0.16,
    margin=dict(l=160, r=25, t=220, b=145),
    font=dict(family="Arial", size=GLOBAL_FONT, color="#1F2933"),
    legend=dict(
        title=dict(text="<b>Model</b>", font=dict(size=LEGEND_TITLE_FONT)),
        orientation="h",
        x=b_legend_center,
        xanchor="center",
        y=1.075,
        yanchor="top",
        font=dict(size=LEGEND_FONT),
        bgcolor="rgba(255,255,255,0)",
        borderwidth=0,
        traceorder="normal",
        itemsizing="constant",
    ),
)


# ============================================================
# DISPLAY AND EXPORT
# ============================================================
fig.show()

fig.write_image(
    "ROGI_XD_combined_figure.pdf",
    format="pdf",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
)
fig.write_image(
    "ROGI_XD_combined_figure.svg",
    format="svg",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
)
fig.write_image(
    "ROGI_XD_combined_figure.png",
    format="png",
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    scale=2,
)